# LTV PostgreSQL Analysis

This notebook performs business-focused SQL analysis on the final customer LTV and high-priority customer tables stored in PostgreSQL.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus
import os

In [2]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = quote_plus(os.getenv("DB_PASSWORD"))
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database connection created successfully.")

Database connection created successfully.


In [3]:
pd.read_sql(
    """
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'public';
    """,
    engine
)

,table_name
0,telco_customer_churn
1,customer_churn_ltv
2,high_priority_customers


In [4]:
pd.read_sql(
    "SELECT COUNT(*) AS total_customers FROM customer_churn_ltv;",
    engine
)

,total_customers
0,7032


In [5]:
pd.read_sql(
    "SELECT COUNT(*) AS high_priority_customers FROM high_priority_customers;",
    engine
)

,high_priority_customers
0,254


*Both PostgreSQL tables were verified successfully. The main customer analytics table contains all customers, while the high-priority table contains only customers classified as High Value - High Risk.*

In [16]:
customer_ltv_df = pd.read_csv("../data/processed/customer_churn_ltv_final.csv")
high_priority_df = pd.read_csv("../data/processed/high_priority_customers.csv")

In [17]:
customer_ltv_df.columns[:5], high_priority_df.columns[:5]

(Index(['Customer_ID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents'], dtype='str'),
 Index(['Customer_ID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents'], dtype='str'))

In [18]:
# Replace PostgreSQL tables with updated versions
customer_ltv_df.to_sql(
    "customer_churn_ltv",
    engine,
    if_exists="replace",
    index=False
)

high_priority_df.to_sql(
    "high_priority_customers",
    engine,
    if_exists="replace",
    index=False
)

print("Updated PostgreSQL tables with Customer_ID successfully.")

Updated PostgreSQL tables with Customer_ID successfully.


In [20]:
pd.read_sql(
    """
    SELECT "Customer_ID", "Estimated_LTV", "Customer_Priority"
    FROM high_priority_customers
    LIMIT 2;
    """,
    engine
)

,Customer_ID,Estimated_LTV,Customer_Priority
0,0280-XJGEX,5081.30,High Value - High Risk
1,6467-CHFZW,4669.45,High Value - High Risk


In [21]:
# Customer count by ltv segment
pd.read_sql(
    """
    SELECT 
        "LTV_Segment",
        COUNT(*) AS customer_count
    FROM customer_churn_ltv
    GROUP BY "LTV_Segment"
    ORDER BY customer_count DESC;
    """,
    engine
)

,LTV_Segment,customer_count
0,Low Value,3516
1,High Value,1758
2,Medium Value,1758


In [7]:
# Average LTV by customer priority
pd.read_sql(
    """
    SELECT 
        "Customer_Priority",
        COUNT(*) AS customer_count,
        ROUND(AVG("Estimated_LTV")::numeric, 2) AS average_ltv
    FROM customer_churn_ltv
    GROUP BY "Customer_Priority"
    ORDER BY average_ltv DESC;
    """,
    engine
)

,Customer_Priority,customer_count,average_ltv
0,High Value - Low Risk,1504,5747.11
1,High Value - High Risk,254,5534.91
2,Medium Value - High Risk,405,2422.89
3,Medium Value - Low Risk,1353,2393.33
4,Low Value - Low Risk,2306,568.38
5,Low Value - High Risk,1210,392.93


In [9]:
# High-priority customers summary
pd.read_sql(
    """
    SELECT 
        COUNT(*) AS high_priority_customers,
        ROUND(AVG("Estimated_LTV")::numeric, 2) AS average_ltv,
        ROUND(SUM("Estimated_LTV")::numeric, 2) AS total_ltv_at_risk
    FROM high_priority_customers;
    """,
    engine
)

,high_priority_customers,average_ltv,total_ltv_at_risk
0,254,5534.91,1405866.15


In [10]:
# Top 10 high-priority customers by LTV
pd.read_sql(
    """
    SELECT 
        "Estimated_LTV",
        "MonthlyCharges",
        "tenure",
        "Customer_Priority"
    FROM high_priority_customers
    ORDER BY "Estimated_LTV" DESC
    LIMIT 10;
    """,
    engine
)

,Estimated_LTV,MonthlyCharges,tenure,Customer_Priority
0,8481.60,117.80,72,High Value - High Risk
1,8095.50,115.65,70,High Value - High Risk
2,8088.50,115.55,70,High Value - High Risk
3,7994.00,114.20,70,High Value - High Risk
4,7929.45,118.35,67,High Value - High Risk
5,7866.00,109.25,72,High Value - High Risk
6,7785.40,116.20,67,High Value - High Risk
7,7710.60,108.60,71,High Value - High Risk
8,7694.20,113.15,68,High Value - High Risk
9,7671.55,108.05,71,High Value - High Risk


In [11]:
# Revenue Value by Customer Priority
revenue_by_priority = pd.read_sql(
    """
    SELECT
        "Customer_Priority",
        COUNT(*) AS customer_count,
        ROUND(SUM("Estimated_LTV")::numeric, 2) AS total_ltv_value
    FROM customer_churn_ltv
    GROUP BY "Customer_Priority"
    ORDER BY total_ltv_value DESC;
    """,
    engine
)

revenue_by_priority

,Customer_Priority,customer_count,total_ltv_value
0,High Value - Low Risk,1504,8643656.95
1,Medium Value - Low Risk,1353,3238172.55
2,High Value - High Risk,254,1405866.15
3,Low Value - Low Risk,2306,1310685.05
4,Medium Value - High Risk,405,981269.35
5,Low Value - High Risk,1210,475441.40


**Observation:**

- The High Value - Low Risk segment contributes the highest total customer value, indicating that a significant portion of business revenue comes from stable high-value customers.

- The High Value - High Risk segment contains only 254 customers but represents more than 1.4 million in estimated customer value. This makes it one of the most critical customer groups for retention efforts, as losing these customers would have a substantial business impact.

- The analysis highlights that customer value is more important than customer count when prioritizing retention strategies.

In [12]:
# LTV at risk by segment
ltv_at_risk_by_segment = pd.read_sql(
    """
    SELECT
        "LTV_Segment",
        COUNT(*) AS churned_customers,
        ROUND(SUM("Estimated_LTV")::numeric, 2) AS ltv_at_risk,
        ROUND(AVG("Estimated_LTV")::numeric, 2) AS avg_ltv_at_risk
    FROM customer_churn_ltv
    WHERE "Churn" = 1
    GROUP BY "LTV_Segment"
    ORDER BY ltv_at_risk DESC;
    """,
    engine
)

ltv_at_risk_by_segment

,LTV_Segment,churned_customers,ltv_at_risk,avg_ltv_at_risk
0,High Value,254,1405866.15,5534.91
1,Medium Value,405,981269.35,2422.89
2,Low Value,1210,475441.40,392.93


**Observation:**

- The High Value segment represents the highest revenue exposure despite having the fewest churned customers. A total estimated customer value of more than 1.4 million is associated with churned High Value customers, compared to approximately 475 thousand for the Low Value segment.

- This finding demonstrates that customer value is a stronger indicator of business impact than customer count. Retention strategies should prioritize High Value customers because the loss of a small number of these customers can have a significantly greater financial impact than the loss of a large number of Low Value customers.

In [15]:
# Retention Priority Summary
retention_priority_summary = pd.read_sql(
    """
    SELECT
        "Customer_Priority",
        COUNT(*) AS customers,
        ROUND(AVG("MonthlyCharges")::numeric, 2) AS avg_monthly_charges,
        ROUND(AVG("tenure")::numeric, 2) AS avg_tenure,
        ROUND(AVG("Estimated_LTV")::numeric, 2) AS avg_ltv
    FROM customer_churn_ltv
    GROUP BY "Customer_Priority"
    ORDER BY avg_ltv DESC;
    """,
    engine
)

retention_priority_summary

,Customer_Priority,customers,avg_monthly_charges,avg_tenure,avg_ltv
0,High Value - Low Risk,1504,92.63,62.23,5747.11
1,High Value - High Risk,254,99.49,55.66,5534.91
2,Medium Value - High Risk,405,84.71,29.80,2422.89
3,Medium Value - Low Risk,1353,63.98,42.39,2393.33
4,Low Value - Low Risk,2306,39.31,18.84,568.38
5,Low Value - High Risk,1210,65.75,6.11,392.93


**Observation:**

- The retention priority summary shows clear differences between customer groups. High Value - Low Risk customers have the highest average LTV, showing that they are long-term and valuable customers who are currently retained.

- The High Value - High Risk group is the most important retention segment because these customers have the highest average monthly charges and a high average LTV, but they are still at churn risk. In contrast, Low Value - High Risk customers have very low average tenure and LTV, indicating that many of them are newer customers with lower overall revenue contribution.

- This analysis helps the business prioritize retention efforts based on customer value, spending behaviour, and relationship duration.

In [22]:
# Top 10 Highest LTV Customers at Risk
top_10_high_risk_customers = pd.read_sql(
    """
    SELECT
        "Customer_ID",
        "Estimated_LTV",
        "MonthlyCharges",
        "tenure",
        "Customer_Priority"
    FROM high_priority_customers
    ORDER BY "Estimated_LTV" DESC
    LIMIT 10;
    """,
    engine
)

top_10_high_risk_customers

,Customer_ID,Estimated_LTV,MonthlyCharges,tenure,Customer_Priority
0,2889-FPWRM,8481.60,117.80,72,High Value - High Risk
1,1444-VVSGW,8095.50,115.65,70,High Value - High Risk
2,0201-OAMXR,8088.50,115.55,70,High Value - High Risk
3,1555-DJEQW,7994.00,114.20,70,High Value - High Risk
4,8199-ZLLSA,7929.45,118.35,67,High Value - High Risk
5,3886-CERTZ,7866.00,109.25,72,High Value - High Risk
6,9053-JZFKV,7785.40,116.20,67,High Value - High Risk
7,7317-GGVPB,7710.60,108.60,71,High Value - High Risk
8,5271-YNWVR,7694.20,113.15,68,High Value - High Risk
9,2834-JRTUA,7671.55,108.05,71,High Value - High Risk


**Observation:**

- The Top 10 High-Risk Customers analysis identifies the most valuable customers currently at risk of churn. These customers have high monthly charges, long customer tenure, and the highest estimated lifetime value.

- The inclusion of Customer_ID enables customer-level retention actions and allows business teams to directly target customers who contribute significant revenue but are at risk of leaving.

In [23]:
# Segment-Wise Churn and Revenue Impact
segment_churn_revenue_impact = pd.read_sql(
    """
    SELECT
        "LTV_Segment",
        COUNT(*) AS total_customers,
        SUM("Churn") AS churned_customers,
        ROUND((SUM("Churn")::numeric / COUNT(*)) * 100, 2) AS churn_rate_percentage,
        ROUND(SUM("Estimated_LTV")::numeric, 2) AS total_ltv_value,
        ROUND(SUM(CASE WHEN "Churn" = 1 THEN "Estimated_LTV" ELSE 0 END)::numeric, 2) AS churned_ltv_value
    FROM customer_churn_ltv
    GROUP BY "LTV_Segment"
    ORDER BY churned_ltv_value DESC;
    """,
    engine
)

segment_churn_revenue_impact

,LTV_Segment,total_customers,churned_customers,churn_rate_percentage,total_ltv_value,churned_ltv_value
0,High Value,1758,254.0,14.45,10049523.10,1405866.15
1,Medium Value,1758,405.0,23.04,4219441.90,981269.35
2,Low Value,3516,1210.0,34.41,1786126.45,475441.40


**Observation:**

- The High Value segment contributes the largest share of total customer value, exceeding 10 million in estimated LTV. Although its churn rate is lower than other segments, it accounts for the highest revenue exposure, with more than 1.4 million in churned customer value.

- This finding demonstrates that revenue impact should be considered alongside churn rate when designing customer retention strategies.

In [24]:
# Best customers to Retain First
best_customers_to_retain = pd.read_sql(
    """
    SELECT
        "Customer_ID",
        "Estimated_LTV",
        "MonthlyCharges",
        "tenure",
        "LTV_Segment",
        "Customer_Priority"
    FROM high_priority_customers
    ORDER BY "Estimated_LTV" DESC, "MonthlyCharges" DESC
    LIMIT 20;
    """,
    engine
)

best_customers_to_retain

,Customer_ID,Estimated_LTV,MonthlyCharges,tenure,LTV_Segment,Customer_Priority
0,2889-FPWRM,8481.60,117.80,72,High Value,High Value - High Risk
1,1444-VVSGW,8095.50,115.65,70,High Value,High Value - High Risk
2,0201-OAMXR,8088.50,115.55,70,High Value,High Value - High Risk
3,1555-DJEQW,7994.00,114.20,70,High Value,High Value - High Risk
4,8199-ZLLSA,7929.45,118.35,67,High Value,High Value - High Risk
5,3886-CERTZ,7866.00,109.25,72,High Value,High Value - High Risk
6,9053-JZFKV,7785.40,116.20,67,High Value,High Value - High Risk
7,7317-GGVPB,7710.60,108.60,71,High Value,High Value - High Risk
8,5271-YNWVR,7694.20,113.15,68,High Value,High Value - High Risk
9,2834-JRTUA,7671.55,108.05,71,High Value,High Value - High Risk


**Observation:**

- The retention target list highlights customers with the highest estimated lifetime value and monthly spending. All identified customers belong to the High Value - High Risk segment, confirming that the customer prioritization framework successfully identifies customers with the greatest potential business impact.

- These customers should be considered the highest priority for retention campaigns and proactive customer engagement strategies.

## Final Business Summary

### Key Findings

| Insight | Result |
|----------|----------|
| Total Customers Analysed | 7,032 |
| High-Priority Customers | 254 |
| Highest Value Segment | High Value - Low Risk |
| Most Critical Risk Segment | High Value - High Risk |
| Total LTV at Risk (High Value Customers) | 1.4 Million+ |
| Highest Churn Rate | Low Value Segment (34.41%) |
| Highest Revenue Exposure | High Value Segment |

### Business Insights

- High Value customers contribute the largest share of total customer value.
- Although High Value customers have a lower churn rate, they represent the highest financial risk when they leave.
- High Value - High Risk customers should be the primary focus of retention campaigns.
- Customer value is a stronger indicator of business impact than customer count.
- A small number of high-value customer losses can create significantly larger revenue loss than a large number of low-value customer losses.
- Customer_ID restoration enables customer-level targeting and business action.

### Recommended Retention Strategy

#### Priority 1

High Value - High Risk Customers

- Immediate retention campaigns
- Customer success outreach
- Loyalty incentives
- Personalized offers

#### Priority 2

Medium Value - High Risk Customers

- Targeted engagement campaigns
- Contract renewal monitoring

#### Priority 3

Low Value - High Risk Customers

- Automated retention campaigns
- Cost-efficient communication strategies

### Conclusion

The analysis successfully identified customer segments, quantified revenue exposure, and generated a prioritized retention framework. The High Value - High Risk segment represents the most important business opportunity for reducing churn and protecting long-term revenue.

## Technical Notes

### Libraries Used

| Library | Purpose |
|----------|----------|
| pandas | Data loading, processing, and SQL result analysis |
| SQLAlchemy | PostgreSQL database connection |
| psycopg2 | PostgreSQL database driver |
| python-dotenv | Secure loading of environment variables |
| urllib.parse.quote_plus | Encoding database passwords containing special characters |

### SQL Functions Used

| Function | Purpose |
|----------|----------|
| COUNT() | Count customers and records |
| SUM() | Calculate total LTV values |
| AVG() | Calculate average customer metrics |
| ROUND() | Improve readability of numeric outputs |
| GROUP BY | Segment customers into business categories |
| ORDER BY | Rank segments and customers by importance |
| CASE WHEN | Conditional calculations for churn impact |
| LIMIT | Retrieve top customers and business targets |

### PostgreSQL Tables Used

| Table | Purpose |
|----------|----------|
| telco_customer_churn | Cleaned customer data |
| customer_churn_ltv | Final customer analytics table |
| high_priority_customers | High Value - High Risk customers |

### Outputs Generated

- Customer Segmentation Analysis
- Revenue Exposure Analysis
- LTV Risk Analysis
- Retention Priority Analysis
- High-Priority Customer Identification
- Customer-Level Retention Lists